# Clean Notebook

In [1]:
import torch
import numpy as np

## Load/ Preprocess Data

In [2]:
"""
Provided Code From PTB Authors
"""
import pandas as pd
import numpy as np
import wfdb
import ast

#provided function to load data with information in the agg_df (modified to return a numpy array)
def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data]).astype(np.float32)
    return data

# path to dta and selected sampling rate
path = "physionet.org/files/ptb-xl/1.0.3/"
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x)) #changes class distribution to dict

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1] #only take diagnostic scps

classes = agg_df.index.tolist()
sub_classes = agg_df.diagnostic_subclass.unique().tolist()
super_classes = agg_df.diagnostic_class.unique().tolist()


"""
Start of Our Code
"""


# encodes the labeled as a 44d vector of 1's and zeros (based off of classes list)
def encode_multilabel(class_list):
    return np.array([1 if cls in class_list else 0 for cls in classes])

# encodes the labeled as a 23d vector of 1's and zeros (based off of sub classes list)
def encode_multilabel_sub(class_list):
    return np.array([1 if cls in class_list else 0 for cls in sub_classes])

# encodes the labeled as a 5d vector of 1's and zeros (based off of sub classes list)
def encode_multilabel_super(class_list):
    return np.array([1 if cls in class_list else 0 for cls in super_classes])

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 30:
                tmp.append(key) #diagnostic, diagnostic_subclass #agg_df.loc[key].index
    return list(set(tmp))

def aggregate_diagnostic_sub(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 50:
                tmp.append(agg_df.loc[key].diagnostic_subclass)
    return list(set(tmp))

def aggregate_diagnostic_super(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 50:
                tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

# Apply diagnostic superclass and encoding multilabel
Y['indv_diagnostic'] = Y.scp_codes.apply(aggregate_diagnostic).apply(encode_multilabel)
Y['indv_diagnostic_str'] = str(Y['indv_diagnostic'])

Y['sub_diagnostic'] = Y.scp_codes.apply(aggregate_diagnostic_sub).apply(encode_multilabel_sub)
Y['sub_diagnostic_str'] = str(Y['sub_diagnostic'])

Y['super_diagnostic'] = Y.scp_codes.apply(aggregate_diagnostic_super).apply(encode_multilabel_super)
Y['super_diagnostic_str'] = str(Y['super_diagnostic'])


## Data Prep

In [3]:
from sklearn.model_selection import train_test_split

#stratified sample

Y_transformed = np.stack(Y.indv_diagnostic.values).astype(np.float32)

# Load raw signal data
X = load_raw_data(Y, sampling_rate, path)

X = np.transpose(X, (0, 2, 1))

X_train, X_test, y_train, y_test = train_test_split(X, Y_transformed, test_size=0.1, random_state=56)

In [4]:
X_train_torch = torch.tensor(X_train) # change dim ordering for PyTorch
y_train_torch = torch.tensor(y_train)
X_test_torch = torch.tensor(X_test) # change dim ordering for PyTorch
y_test_torch = torch.tensor(y_test)

In [5]:
from torch.utils.data import WeightedRandomSampler

condition_freq = y_train_torch.sum(axis=0)
condition_inv_freq = 1.0 / (np.log(condition_freq + 1e-6))

row_weights = (y_train_torch * condition_inv_freq).sum(axis=1)
row_weights = row_weights / row_weights.sum()
sampler = WeightedRandomSampler(
    weights=row_weights,
    num_samples=len(row_weights),
    replacement = True
)

/tmp/ipykernel_3328/380670768.py:4: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  condition_inv_freq = 1.0 / (np.log(condition_freq + 1e-6))


In [6]:
train_dataset = torch.utils.data.TensorDataset(X_train_torch, y_train_torch)
train_dataloader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
weighted_train_dataloader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=32, sampler=sampler)

test_dataset = torch.utils.data.TensorDataset(X_test_torch, y_test_torch)
test_dataloader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=32)


## CNN

In [7]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class CNet(nn.Module):
  def __init__(self):
    super(CNet,self).__init__()

    self.conv11 = nn.Conv1d(12, 64, 9, padding=4)
    self.conv12 = nn.Conv1d(64, 64, 9, padding=4)
    self.bn1 = nn.BatchNorm1d(64)

    self.pool = nn.MaxPool1d(kernel_size=2, stride = 2)

    self.conv21 = nn.Conv1d(64, 128, 9, padding=4)
    self.conv22 = nn.Conv1d(128, 128, 9, padding=4)
    self.bn2 = nn.BatchNorm1d(128)

    self.conv31 = nn.Conv1d(128, 256, 9, padding=4)
    self.conv32 = nn.Conv1d(256, 256, 9, padding=4)
    self.conv33 = nn.Conv1d(256, 256, 9, padding=4)
    self.bn3 = nn.BatchNorm1d(256)

    self.conv41 = nn.Conv1d(256, 512, 9, padding=4)
    self.conv42 = nn.Conv1d(512, 512, 9, padding=4)
    self.conv43 = nn.Conv1d(512, 512, 9, padding=4)
    self.bn4 = nn.BatchNorm1d(512)

    self.fc1 = nn.Linear(512 * 125, 1000)
    self.fc2 = nn.Linear(1000, 500)
    self.fc3 = nn.Linear(500, 44)

  def forward(self, x):
    x = self.pool(F.relu(self.bn1(self.conv12(F.relu(self.conv11(x))))))
    x = self.pool(F.relu(self.bn2(self.conv22(F.relu(self.conv21(x))))))
    x = self.pool(F.relu(self.bn3(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x))))))))
    x = F.relu(self.bn4(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x)))))))
    x = torch.flatten(x,1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

In [8]:
class ResidualBlock1D(nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size=9, dropout=0.1):
    super().__init__()
    padding = kernel_size // 2

    self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
    self.bn1 = nn.BatchNorm1d(out_channels)

    self.conv12 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
    self.bn2 = nn.BatchNorm1d(out_channels)

    self.dropout = nn.Dropout(dropout)

    if in_channels != out_channels:
      #residual would be uneven here
      self.shortcut = nn.Conv1d(in_channels, out_channels, kernel_size=1) # reduce the channels
    else:
      self.shortcut = nn.Identity()

  def forward(self, x):
      identity = self.shortcut(x) #residual

      x = self.conv1(x)
      x = self.bn1(x)
      x = F.relu(x)

      x = self.dropout(x)

      x = self.conv12(x)
      x = self.bn2(x)
      
      x = x + identity

      x = F.relu(x)

      return x
    
class CINet(nn.Module):
  def __init__(self):
    super(CINet,self).__init__()
    self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
    
    self.block1 = ResidualBlock1D(12, 64, kernel_size=21, dropout=0.1)

    self.block2 = ResidualBlock1D(64, 128, kernel_size=21, dropout=0.1)

    self.block3 = ResidualBlock1D(128, 256, kernel_size=21, dropout=0.2)

    self.block4 = ResidualBlock1D(256, 512, kernel_size=21, dropout=0.2)

    self.gap = nn.AdaptiveAvgPool1d(1)
    
    self.drop_fc = nn.Dropout(0.4)
    self.fc1 = nn.Linear(512, 128)
    self.fc2 = nn.Linear(128, 44)

  def forward(self, x):

    x = self.block1(x)
    x = self.pool(x)

    x = self.block2(x)
    x = self.pool(x)

    x = self.block3(x)
    x = self.pool(x)

    x = self.block4(x)

    x = self.gap(x)
    x = x.squeeze(-1)

    x = self.drop_fc(x)
    x = F.relu(self.fc1(x))
    x = self.drop_fc(x)
    
    x = self.fc2(x)
    return x
    

In [9]:
def weighted_train_model(model, train_dataloader, test_dataloader, epochs, weight_decay):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.train()

    f = torch.tensor(y_train).sum(dim=0)
    N = y_train.shape[0]
    pos_weight = (N - f) / (f + 1e-6)
    pos_weight = torch.log(pos_weight.to(device))
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))

    lr = 1e-3
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_test = 0
    best_test_train = 0
    best_epoch = 0

    for epoch in range(epochs):
        running_loss = 0
        for X, y in train_dataloader:
            X,y = X.to(device), y.to(device)

            opt.zero_grad() #zero out the gradients

            z = model(X)
            loss = loss_fn(z,y) #compute loss
            running_loss += loss.item()

            loss.backward() #compute gradients

            opt.step() #apply gradients
        scheduler.step()



        running_loss = running_loss / len(train_dataloader)
        print(f"Epoch {epoch} train loss: {running_loss}")

def train_model(model, train_dataloader, test_dataloader, epochs, weight_decay):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.train()

    loss_fn = torch.nn.BCEWithLogitsLoss()

    lr = 1e-3
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_test = 0
    best_test_train = 0
    best_epoch = 0

    for epoch in range(epochs):
        running_loss = 0
        for X, y in train_dataloader:
            X,y = X.to(device), y.to(device)

            opt.zero_grad() #zero out the gradients

            z = model(X)
            loss = loss_fn(z,y) #compute loss
            running_loss += loss.item()

            loss.backward() #compute gradients

            opt.step() #apply gradients
        scheduler.step()



        running_loss = running_loss / len(train_dataloader)
        print(f"Epoch {epoch} train loss: {running_loss}")


In [10]:
device = "cuda"

In [47]:
from sklearn.metrics import jaccard_score

def macro_precision(y_pred, y_true):
    y_pred = y_pred.detach().to("cpu", dtype=torch.int32)
    y_true = y_true.detach().to("cpu", dtype=torch.int32)
    t_p = ((y_pred == 1) & (y_true == 1)).sum(dim=0)
    f_p = ((y_pred == 1) & (y_true == 0)).sum(dim=0)
    return t_p.float() / (t_p + f_p).float().clamp(min=1e-8)

def macro_recall(y_pred, y_true):
    y_pred = y_pred.detach().to("cpu", dtype=torch.int32)
    y_true = y_true.detach().to("cpu", dtype=torch.int32)
    t_p = ((y_pred == 1) & (y_true == 1)).sum(dim=0)
    f_n = ((y_pred == 0) & (y_true == 1)).sum(dim=0)
    return t_p.float() / (t_p + f_n).float().clamp(min=1e-8)

def macro_f1(y_pred, y_true):
    m_p = macro_precision(y_pred, y_true)
    m_r = macro_recall(y_pred, y_true)
    return 2 * m_p * m_r / (m_p + m_r).clamp(min=1e-8)

def dig_accurate(y_preds, y_ground):
    y_preds_cpu = y_preds.detach().to("cpu", dtype=torch.int32)
    y_ground_cpu = y_ground.detach().to("cpu", dtype=torch.int32)
    element_wise_eq = y_preds_cpu == y_ground_cpu
    y_preds = y_preds_cpu.numpy()
    y_ground = y_ground_cpu.numpy()

    row_acc = torch.all(element_wise_eq, dim=1).float().mean().item()
    elem_acc = element_wise_eq.float().mean().item()
    return jaccard_score(y_ground, y_preds, average="samples", zero_division=0), row_acc, elem_acc

def optim_jaccard_threshold(model):
    model.eval()

    with torch.no_grad():
        logits = model(X_test_torch.to(device))
        probs = torch.sigmoid(logits).cpu()

    best_threshold = -1
    best_score = -1

    input = y_test_torch.int()

    # Step by 0.05
    for t in torch.arange(0.0, 1.01, 0.05):
        preds = (probs > t).int()
        jaccard = jaccard_score(preds, input, average="samples", zero_division=0)
        if best_score < jaccard:
            best_score = jaccard
            best_threshold = t

    return best_score, best_threshold

def report_metric(model, threshold=0.5):
    model.eval()
    count = 0

    f1_score = []
    p_score = []
    r_score = []
    jaccard = 0.0
    row_acc = 0.0
    elem_acc = 0.0

    for x_test, y_test in test_dataloader:
        val = (torch.sigmoid(model(x_test.to(device))) > threshold).int()
        count += 1
        y_test = y_test.to(device).int()
        f1_score.append(macro_f1(val, y_test))
        p_score.append(macro_precision(val, y_test))
        r_score.append(macro_recall(val, y_test))
        j, r, e = dig_accurate(val, y_test)
        jaccard += j
        row_acc += r
        elem_acc += e

    f1_score = torch.stack(f1_score).cpu()
    p_score = torch.stack(p_score).cpu()
    r_score = torch.stack(r_score).cpu()
    print(f"Jaccard: {jaccard/count} | Row Accuracy: {row_acc/count} | Element Accuracy {elem_acc/count} | F1 Score {f1_score.mean()} | Precision {p_score.mean()} | Recall {r_score.mean()}")

def report_metrics(model):
    _, best_threshold = optim_jaccard_threshold(model)
    print("Base Threshold Results: ")
    report_metric(model)
    print("Tuned on Jaccard Threshold Results:")
    report_metric(model, best_threshold)
    

In [ ]:
base_model = CNet().to(device)
train_model(base_model, train_dataloader, test_dataloader, 30, 1e-3)

Epoch 0 train loss: 0.10347103261986462
Epoch 1 train loss: 0.08708210347117158
Epoch 2 train loss: 0.08476161199594553
Epoch 3 train loss: 0.08312258425287004
Epoch 4 train loss: 0.08247327780903162
Epoch 5 train loss: 0.08174097157577932
Epoch 6 train loss: 0.08135958466715261
Epoch 7 train loss: 0.08097154166585847
Epoch 8 train loss: 0.08056533552633435
Epoch 9 train loss: 0.0802154902198901
Epoch 10 train loss: 0.08014822943615409
Epoch 11 train loss: 0.07979539179937847
Epoch 12 train loss: 0.07960441262015303
Epoch 13 train loss: 0.07939234408716814
Epoch 14 train loss: 0.07887338733464383
Epoch 15 train loss: 0.07862487735062934
Epoch 16 train loss: 0.0784740085055269
Epoch 17 train loss: 0.07827364736470221
Epoch 18 train loss: 0.07782516588459962
Epoch 19 train loss: 0.07753569129187043
Epoch 20 train loss: 0.07729898671034105
Epoch 21 train loss: 0.07707898538401926
Epoch 22 train loss: 0.07658692324952698
Epoch 23 train loss: 0.0766068856605004
Epoch 24 train loss: 0.076261

In [ ]:
weighted_samp_model = CNet().to(device)
train_model(weighted_samp_model, weighted_train_dataloader, test_dataloader, 30, 1e-3)

Epoch 0 train loss: 0.14450159446861147
Epoch 1 train loss: 0.12081639839985471
Epoch 2 train loss: 0.11879336790269672
Epoch 3 train loss: 0.11580065781013973
Epoch 4 train loss: 0.11343463835479382
Epoch 5 train loss: 0.11156004782559817
Epoch 6 train loss: 0.11079492793700595
Epoch 7 train loss: 0.10941338773319309
Epoch 8 train loss: 0.10829993400492187
Epoch 9 train loss: 0.10751299453286473
Epoch 10 train loss: 0.10577034688151232
Epoch 11 train loss: 0.10522925130513282
Epoch 12 train loss: 0.1065108299570868
Epoch 13 train loss: 0.10482749109936071
Epoch 14 train loss: 0.103094423207961
Epoch 15 train loss: 0.10219203623721576
Epoch 16 train loss: 0.10165716252056123
Epoch 17 train loss: 0.10100069774332963
Epoch 18 train loss: 0.10222657875488558
Epoch 19 train loss: 0.0995937277371798
Epoch 20 train loss: 0.0996591692323211
Epoch 21 train loss: 0.09881200073499245
Epoch 22 train loss: 0.09832499836987121
Epoch 23 train loss: 0.09903986057333333
Epoch 24 train loss: 0.09784388

In [ ]:
weighted_loss_model = CNet().to(device)
weighted_train_model(weighted_loss_model, train_dataloader, test_dataloader, 30, 1e-3)

/tmp/ipykernel_3328/3598958252.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))


Epoch 0 train loss: 0.22021155999154338
Epoch 1 train loss: 0.19153565172021086
Epoch 2 train loss: 0.185795855395957
Epoch 3 train loss: 0.18220967547027606
Epoch 4 train loss: 0.18093799730514082
Epoch 5 train loss: 0.1778295691510365
Epoch 6 train loss: 0.17609499399747444
Epoch 7 train loss: 0.17031452057318888
Epoch 8 train loss: 0.1652877539857397
Epoch 9 train loss: 0.16074094794544996
Epoch 10 train loss: 0.15788114595102565
Epoch 11 train loss: 0.15633450595340434
Epoch 12 train loss: 0.15457665372991794
Epoch 13 train loss: 0.15257556264437372
Epoch 14 train loss: 0.15171378689548093
Epoch 15 train loss: 0.150598151032913
Epoch 16 train loss: 0.14924131453959483
Epoch 17 train loss: 0.14822985781569045
Epoch 18 train loss: 0.14716566185366836
Epoch 19 train loss: 0.145710790184014
Epoch 20 train loss: 0.1446426367987833
Epoch 21 train loss: 0.14339326146128512
Epoch 22 train loss: 0.1419217184683496
Epoch 23 train loss: 0.14072726965435942
Epoch 24 train loss: 0.1400235962814

In [ ]:
weighted_loss_samp_model = CNet().to(device)
weighted_train_model(weighted_loss_samp_model, weighted_train_dataloader, test_dataloader, 30, 1e-3)

/tmp/ipykernel_3328/3598958252.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))


Epoch 0 train loss: 0.33356682694113604
Epoch 1 train loss: 0.2806928529364667
Epoch 2 train loss: 0.27173111825598956
Epoch 3 train loss: 0.2683729781050247
Epoch 4 train loss: 0.2585843091145012
Epoch 5 train loss: 0.252207301751216
Epoch 6 train loss: 0.24934179583376315
Epoch 7 train loss: 0.24863047815301131
Epoch 8 train loss: 0.24286402298398438
Epoch 9 train loss: 0.24345612979770095
Epoch 10 train loss: 0.24314017280790629
Epoch 11 train loss: 0.2385390250904164
Epoch 12 train loss: 0.2348565575985256
Epoch 13 train loss: 0.22663109074877605
Epoch 14 train loss: 0.22542584478758057
Epoch 15 train loss: 0.2226689096770768
Epoch 16 train loss: 0.2176977726312336
Epoch 17 train loss: 0.21528181402127983
Epoch 18 train loss: 0.21276562084543976
Epoch 19 train loss: 0.21206437954102744
Epoch 20 train loss: 0.20714785971645425
Epoch 21 train loss: 0.20551712692075133
Epoch 22 train loss: 0.20536400651990008
Epoch 23 train loss: 0.2031877115368843
Epoch 24 train loss: 0.2009937750108

In [ ]:
inet = CINet().to(device)
weighted_train_model(inet, train_dataloader, test_dataloader, 30, 1e-3)

/tmp/ipykernel_3328/3598958252.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))


Epoch 0 train loss: 0.20328532105936678
Epoch 1 train loss: 0.19023666857265495
Epoch 2 train loss: 0.18933364564510433
Epoch 3 train loss: 0.18748416994313075
Epoch 4 train loss: 0.18503839327234786
Epoch 5 train loss: 0.1831506067456174
Epoch 6 train loss: 0.18173587037074448
Epoch 7 train loss: 0.180168509228513
Epoch 8 train loss: 0.17920493372148721
Epoch 9 train loss: 0.1779950288418644
Epoch 10 train loss: 0.17672629203004248
Epoch 11 train loss: 0.1758349027108486
Epoch 12 train loss: 0.175309627423069
Epoch 13 train loss: 0.17415010606936795
Epoch 14 train loss: 0.17381336117703286
Epoch 15 train loss: 0.17202837503718438
Epoch 16 train loss: 0.171578581405482
Epoch 17 train loss: 0.17118035764084577
Epoch 18 train loss: 0.16987420061560718
Epoch 19 train loss: 0.16962641393856817
Epoch 20 train loss: 0.16931467108938128
Epoch 21 train loss: 0.1682116183655561
Epoch 22 train loss: 0.16717979264618518
Epoch 23 train loss: 0.16710733788555143
Epoch 24 train loss: 0.1663417643609

In [ ]:
weighted_samp_inet = CINet().to(device)
train_model(weighted_samp_inet, weighted_train_dataloader, test_dataloader, 30, 1e-3)

Epoch 0 train loss: 0.13731382253309804
Epoch 1 train loss: 0.12431321803977902
Epoch 2 train loss: 0.12330788872674933
Epoch 3 train loss: 0.12080155928233936
Epoch 4 train loss: 0.11810923430838104
Epoch 5 train loss: 0.11627007363188151
Epoch 6 train loss: 0.11535450278596303
Epoch 7 train loss: 0.11575300035336895
Epoch 8 train loss: 0.11503782332040588
Epoch 9 train loss: 0.11493702265600816
Epoch 10 train loss: 0.11229084581379394
Epoch 11 train loss: 0.11327242159591047
Epoch 12 train loss: 0.11242186574186487
Epoch 13 train loss: 0.11264267571100584
Epoch 14 train loss: 0.11194193316471887
Epoch 15 train loss: 0.11073559794441497
Epoch 16 train loss: 0.11145538471469274
Epoch 17 train loss: 0.10999279516576167
Epoch 18 train loss: 0.11037860207672227
Epoch 19 train loss: 0.1094857059686114
Epoch 20 train loss: 0.10888386778654803
Epoch 21 train loss: 0.10721187100591023
Epoch 22 train loss: 0.1081577163759971
Epoch 23 train loss: 0.10694394159928595
Epoch 24 train loss: 0.10658

In [ ]:
weighted_loss_inet = CINet().to(device)
weighted_train_model(weighted_loss_inet, train_dataloader, test_dataloader, 30, 1e-3)

/tmp/ipykernel_3328/3598958252.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))


Epoch 0 train loss: 0.19898818519439682
Epoch 1 train loss: 0.1826178007774322
Epoch 2 train loss: 0.18200149405322943
Epoch 3 train loss: 0.17514903762101738
Epoch 4 train loss: 0.1716587347515437
Epoch 5 train loss: 0.1702060332030349
Epoch 6 train loss: 0.16654243451279227
Epoch 7 train loss: 0.1643293088465832
Epoch 8 train loss: 0.1634484521001092
Epoch 9 train loss: 0.16102501774261363
Epoch 10 train loss: 0.15996589801647376
Epoch 11 train loss: 0.15842252046306668
Epoch 12 train loss: 0.15753126780256774
Epoch 13 train loss: 0.1567117213863115
Epoch 14 train loss: 0.15659232779803417
Epoch 15 train loss: 0.15538157704695815
Epoch 16 train loss: 0.1537231835703508
Epoch 17 train loss: 0.1529017203769583
Epoch 18 train loss: 0.1516179825855777
Epoch 19 train loss: 0.15160587553772165
Epoch 20 train loss: 0.1506018061494206
Epoch 21 train loss: 0.14891365802079343
Epoch 22 train loss: 0.1479533553511778
Epoch 23 train loss: 0.14710635144615405
Epoch 24 train loss: 0.14590281231711

In [ ]:
weighted_samp_loss_inet = CINet().to(device)
weighted_train_model(weighted_samp_loss_inet, weighted_train_dataloader, test_dataloader, 30, 1e-3)

/tmp/ipykernel_3328/3598958252.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))


Epoch 0 train loss: 0.3037492003813629
Epoch 1 train loss: 0.2766672185704063
Epoch 2 train loss: 0.2684456276000517
Epoch 3 train loss: 0.2682099479223307
Epoch 4 train loss: 0.2592700589751576
Epoch 5 train loss: 0.25692683749556156
Epoch 6 train loss: 0.2554000273152749
Epoch 7 train loss: 0.25537856805353676
Epoch 8 train loss: 0.2507448742948837
Epoch 9 train loss: 0.24901630724583076
Epoch 10 train loss: 0.24282129358876411
Epoch 11 train loss: 0.24077036621273534
Epoch 12 train loss: 0.24077193742369207
Epoch 13 train loss: 0.23839196812737648
Epoch 14 train loss: 0.2374172109491662
Epoch 15 train loss: 0.2306168311387786
Epoch 16 train loss: 0.2287262371628991
Epoch 17 train loss: 0.22736885138948112
Epoch 18 train loss: 0.22556626512775205
Epoch 19 train loss: 0.2213696734767395
Epoch 20 train loss: 0.21929484350964767
Epoch 21 train loss: 0.21906914656635992
Epoch 22 train loss: 0.21674035394910104
Epoch 23 train loss: 0.21705507317175696
Epoch 24 train loss: 0.21269313304257

In [48]:
print("\nBase VGG16 Style Model\n")
report_metrics(base_model)
print("\nBase Model With Weighted Sampling\n")
report_metrics(weighted_samp_model)
print("\nBase Model With Weighted Loss\n")
report_metrics(weighted_loss_model)
print("\nBase Model With Weighted Sampling And Loss\n")
report_metrics(weighted_loss_samp_model)
print("\nResidual Connections Model With Weighted Sampling\n")
report_metrics(weighted_samp_inet)
print("\nResidual Connections Model With Weighted Loss\n")
report_metrics(weighted_loss_inet)
print("\nResidual Connections Model With Weighted Sampling And Loss\n")
report_metrics(weighted_samp_loss_inet)


Base VGG16 Style Model

Base Threshold Results: 
Jaccard: 0.36356431159420294 | Row Accuracy: 0.3976449275362319 | Element Accuracy 0.9779829529748447 | F1 Score 0.029833173379302025 | Precision 0.031201519072055817 | Recall 0.02947944961488247
Tuned on Jaccard Threshold Results:
Jaccard: 0.4877756930066714 | Row Accuracy: 0.3745471014492754 | Element Accuracy 0.955368905827619 | F1 Score 0.07527042180299759 | Precision 0.05879997834563255 | Recall 0.13160638511180878

Base Model With Weighted Sampling

Base Threshold Results: 
Jaccard: 0.4573973429951691 | Row Accuracy: 0.432518115942029 | Element Accuracy 0.9795783948207247 | F1 Score 0.08176559954881668 | Precision 0.08580842614173889 | Recall 0.08476199209690094
Tuned on Jaccard Threshold Results:
Jaccard: 0.551659190246147 | Row Accuracy: 0.4447463768115942 | Element Accuracy 0.9716629610545393 | F1 Score 0.11527277529239655 | Precision 0.10401813685894012 | Recall 0.14810900390148163

Base Model With Weighted Loss

Base Threshol